In [5]:
!pip install pandas
!pip install chromadb
import chromadb
import pandas as pd
import numpy as np


Текущая кодовая страница: 1251
Текущая кодовая страница: 1251
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
   ---------------------------------------- 0.0/23.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/23.5 MB ? eta -:--:--
   ---------------------------------------- 0.3/23.5 MB ? eta -:--:--
    --------------------------------------- 0.5/23.5 MB 1.1 MB/s eta 0:00:21
   - -------------------------------------- 0.8/23.5 MB 1.1 MB/s eta 0:00:21
   - -------------------------------------- 1.0/23.5 MB 1.1 MB/s eta 0:00:20
   -- ------------------------------------- 1.3/23.5 MB 1.2 MB/s eta 0:00:20
   -- ------------------------------------- 1.6/23.5 MB 1.2 MB/s eta 0:00:19
   --- ------------------------------------ 1.8/23.5 MB 1.2 MB/s eta 0:00:19
   --- ------------------------------------ 2.1/23.5 MB 1.2 MB/s eta 0:00:19
   ---- ----------------------------------- 2.4/23.5 MB

In [6]:
data = pd.read_csv('metadata.csv', usecols = ['title', 'abstract'])
data.head()

data = data.dropna(subset=['abstract'])
data = data.sample(n=100, random_state=42).reset_index(drop=True)
data['text'] = data['title'] + ". " + data['abstract']

In [13]:
!pip install sentence_transformers

Текущая кодовая страница: 1251
  Using cached setuptools-84.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/740.6 kB ? eta -:--:--
   -------------- ------------------------- 262.1/740.6 kB ? eta -:--:--
   ---------------------------------------- 740.6/740.6 kB 3.8 MB/s  0:00:00
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   ----- ---------------------------------- 1.6/12.3 MB 7.6 MB/s eta 0:00:02
   ---------- ----------------------------- 3.1/12.3 MB 8.4 MB/s eta 0:00:02
   ------------------ --------------------- 5.8/12.3 MB 9.8 MB/s eta 0:00:01
   --------------------------- ------------ 8.4/12.3 MB 10.0 MB/s eta 0:00:01
   ------------------------------------- -- 11.5/12.3 MB 11.1 MB/s eta 0:00:01
   ---------------------------------------- 12.3/12.3 MB 11.0 MB/s  0:00:01
   ------------------

In [26]:
CHUNK_SIZE = 50 
OVERLAP_SIZE = 10 
for i in range(10):
    words = data['text'][i].split()
    print(f"Документ {i+1}: {data['title'][i][:70]}...")
    
    for j in range(0, min(150, len(words)), CHUNK_SIZE):
        if j + CHUNK_SIZE < len(words):
            # Берем последние 10 слов текущего куска и первые 10 слов следующего
            end_of_chunk = ' '.join(words[j+ CHUNK_SIZE - OVERLAP_SIZE:j+ CHUNK_SIZE])
            start_of_next = ' '.join(words[j+ CHUNK_SIZE:j+ CHUNK_SIZE + OVERLAP_SIZE])
            print(f" {j+CHUNK_SIZE}: ...{end_of_chunk} __РАЗРЫВ__ {start_of_next}...")
    print("\n")

Документ 1: Maternal and infant outcomes of full-term pregnancy combined with COVI...
 50: ...coronavirus 2 (SARS-CoV-2) that is responsible for coronavirus disease 2019 __РАЗРЫВ__ (COVID-2019). DESIGN: Retrospective case series. SETTING: The Central Hospitals of...
 100: ...College, Huazhong University of Science and Technology, confirmed SARS-CoV-2 infection __РАЗРЫВ__ and COVID-2019 with laboratorial and clinical methods, were reviewed by...
 150: ...form, such as epidemiological history, clinical manifestations, auxiliary examination of __РАЗРЫВ__ major laboratory and clinic, etc. All the information was collected...


Документ 2: COVID-19: Characteristics and Therapeutics...
 50: ...quickly designated as a global pandemic by the World Health __РАЗРЫВ__ Organization as there have been about 98.0 million confirmed cases...
 100: ...tested or are currently under development to mitigate its risk-factors. __РАЗРЫВ__ Recently, some vaccine candidates showed around 95% clinical efficacy,

In [27]:


def fixed_size_chunking(text, chunk_size=50, overlap_size=0):
    words = text.split()
    chunks = []
    step = chunk_size - overlap_size
    
    if step <= 0:
        step = chunk_size

    for i in range(0, len(words), step):
        chunk_words = words[i:i + chunk_size]
        chunk_text = ' '.join(chunk_words)
        chunks.append(chunk_text)
        if i + chunk_size >= len(words):
            break
            
    return chunks


data['chunks_no_overlap'] = data['text'].apply(lambda x: fixed_size_chunking(x, CHUNK_SIZE, 0))
data['chunks_with_overlap'] = data['text'].apply(lambda x: fixed_size_chunking(x, CHUNK_SIZE, OVERLAP_SIZE))

total_no = sum(len(x) for x in data['chunks_no_overlap'])
total_ov = sum(len(x) for x in data['chunks_with_overlap'])

print(f"Всего документов: {len(data)}")
print(f"Без overlap: {total_no} chunks")
print(f"С overlap: {total_ov} chunks")

chroma_client = chromadb.Client()

try:
    chroma_client.delete_collection(name="no_overlap")
except:
    pass

try:
    chroma_client.delete_collection(name="with_overlap")
except:
    pass

collection_no = chroma_client.create_collection(name="no_overlap")
collection_ov = chroma_client.create_collection(name="with_overlap")

def add_chunks_to_collection(collection, chunks_list, prefix):
    ids = []
    documents = []
    for doc_idx, chunks in enumerate(chunks_list):
        for chunk_idx, chunk in enumerate(chunks):
            ids.append(f"{prefix}_doc{doc_idx}_chunk{chunk_idx}")
            documents.append(chunk)
    collection.add(documents=documents, ids=ids)

add_chunks_to_collection(collection_no, data['chunks_no_overlap'].tolist(), "no")
add_chunks_to_collection(collection_ov, data['chunks_with_overlap'].tolist(), "ov")

eval_data = [
    ("What was the study design for the research on coronavirus disease 2019?", "Retrospective case series"),
    ("Which organization designated the coronavirus as a global pandemic?", "World Health Organization"),
    ("What clinical efficacy did vaccine candidates recently show to mitigate risk factors?", "95% clinical efficacy"),
    ("What specimens were collected to detect common childhood respiratory viruses across different age groups?", "Respiratory specimens"),
    ("The epidemiological characteristics of respiratory viruses in 2020 were compared with which year?", "2019"),
    ("What was the study design for evaluating PCR positivity correlations with seroconversion and disease severity?", "Retrospective cohort study"),
    ("What are healthcare systems using repurposing drugs for to cure patients from COVID-19?", "cure the patients"),
    ("Five GABA(A)R mAbs from what fluid cells bound to various epitopes involving the α1?", "cerebrospinal fluid"),
    ("What phenotype with seizures and increased mortality was induced in rodents reminiscent of encephalitis patients' symptoms?", "severe phenotype"),
    ("Why is differential diagnoses for enteritis in adult ruminants not simple based on clinical signs alone?", "zoonotic potential")
]

def search(query, collection, k=3):
    results = collection.query(query_texts=query, n_results=k)
    return results['documents'][0]

K = 3
recall_no = 0
recall_ov = 0


for i, (q, expected_answer) in enumerate(eval_data):
    res_no = search(q, collection_no, K)
    res_ov = search(q, collection_ov, K)
    found_in_no = any(expected_answer.lower() in chunk.lower() for chunk in res_no)
    found_in_ov = any(expected_answer.lower() in chunk.lower() for chunk in res_ov)
    
    print(f"\nВопрос {i+1}: {q}")
    print(f"  Ожидаемый ответ: '{expected_answer}'")
    print(f"  БЕЗ overlap: {'НАЙДЕН' if found_in_no else 'НЕ НАЙДЕН'}")
    print(f"  С overlap:   {'НАЙДЕН' if found_in_ov else 'НЕ НАЙДЕН'}")
    
    if found_in_no: recall_no += 1
    if found_in_ov: recall_ov += 1

recall_no_score = recall_no / len(eval_data)
recall_ov_score = recall_ov / len(eval_data)

print(f"Recall@{K} (БЕЗ overlap): {recall_no_score * 100:.1f}%")
print(f"Recall@{K} (С overlap):   {recall_ov_score * 100:.1f}%")

Всего документов: 100
Без overlap: 507 chunks
С overlap: 593 chunks

Вопрос 1: What was the study design for the research on coronavirus disease 2019?
  Ожидаемый ответ: 'Retrospective case series'
  БЕЗ overlap: НЕ НАЙДЕН
  С overlap:   НЕ НАЙДЕН

Вопрос 2: Which organization designated the coronavirus as a global pandemic?
  Ожидаемый ответ: 'World Health Organization'
  БЕЗ overlap: НЕ НАЙДЕН
  С overlap:   НАЙДЕН

Вопрос 3: What clinical efficacy did vaccine candidates recently show to mitigate risk factors?
  Ожидаемый ответ: '95% clinical efficacy'
  БЕЗ overlap: НЕ НАЙДЕН
  С overlap:   НАЙДЕН

Вопрос 4: What specimens were collected to detect common childhood respiratory viruses across different age groups?
  Ожидаемый ответ: 'Respiratory specimens'
  БЕЗ overlap: НАЙДЕН
  С overlap:   НАЙДЕН

Вопрос 5: The epidemiological characteristics of respiratory viruses in 2020 were compared with which year?
  Ожидаемый ответ: '2019'
  БЕЗ overlap: НАЙДЕН
  С overlap:   НАЙДЕН

Вопрос 6

In [19]:

import re
import numpy as np
import chromadb
from sentence_transformers import SentenceTransformer

st_model = SentenceTransformer('all-MiniLM-L6-v2')

def semantic_chunking(text, percentile=25):
    sentences = re.split(r'(?<=[.!?])\s+', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    
    if len(sentences) <= 1:
        return [text]
        
    embeddings = st_model.encode(sentences)
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    normalized_emb = embeddings / norms
    similarities = np.sum(normalized_emb[:-1] * normalized_emb[1:], axis=1)
    threshold = np.percentile(similarities, percentile)
    
    chunks = []
    current_chunk = [sentences[0]]
    
    for i, sim in enumerate(similarities):
        if sim < threshold:
            chunks.append(' '.join(current_chunk))
            current_chunk = [sentences[i+1]]
        else:
            current_chunk.append(sentences[i+1])
            
    if current_chunk:
        chunks.append(' '.join(current_chunk))
        
    return chunks

data['chunks_semantic'] = data['text'].apply(lambda x: semantic_chunking(x, 25))
total_sem = sum(len(x) for x in data['chunks_semantic'])

print(f"Semantic: {total_sem} chunks")

chroma_client = chromadb.Client()

try:
    chroma_client.delete_collection(name="semantic_chunks")
except:
    pass

collection_sem = chroma_client.create_collection(name="semantic_chunks")

def add_chunks_to_collection(collection, chunks_list, prefix):
    ids = []
    documents = []
    for doc_idx, chunks in enumerate(chunks_list):
        for chunk_idx, chunk in enumerate(chunks):
            ids.append(f"{prefix}_doc{doc_idx}_chunk{chunk_idx}")
            documents.append(chunk)
    collection.add(documents=documents, ids=ids)

add_chunks_to_collection(collection_sem, data['chunks_semantic'].tolist(), "sem")

def search(query, collection, k=3):
    results = collection.query(query_texts=query, n_results=k)
    return results['documents'][0]

K = 3
recall_sem = 0

for i, (q, expected_answer) in enumerate(eval_data):
    res_sem = search(q, collection_sem, K)
    found_in_sem = any(expected_answer.lower() in chunk.lower() for chunk in res_sem)
    
    print(f"\nВопрос {i+1}: {q}")
    print(f"  Ожидаемый ответ: '{expected_answer}'")
    print(f"  Semantic:   {'НАЙДЕН' if found_in_sem else 'НЕ НАЙДЕН'}")
    
    if found_in_sem: recall_sem += 1

recall_sem_score = recall_sem / len(eval_data)
print(f"\nRecall@{K} (Semantic): {recall_sem_score * 100:.1f}%")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Semantic: 330 chunks

Вопрос 1: What was the study design for the research on coronavirus disease 2019?
  Ожидаемый ответ: 'Retrospective case series'
  Semantic:   НЕ НАЙДЕН

Вопрос 2: Which organization designated the coronavirus as a global pandemic?
  Ожидаемый ответ: 'World Health Organization'
  Semantic:   НАЙДЕН

Вопрос 3: What clinical efficacy did vaccine candidates recently show to mitigate risk factors?
  Ожидаемый ответ: '95% clinical efficacy'
  Semantic:   НАЙДЕН

Вопрос 4: What specimens were collected to detect common childhood respiratory viruses across different age groups?
  Ожидаемый ответ: 'Respiratory specimens'
  Semantic:   НАЙДЕН

Вопрос 5: The epidemiological characteristics of respiratory viruses in 2020 were compared with which year?
  Ожидаемый ответ: '2019'
  Semantic:   НАЙДЕН

Вопрос 6: What was the study design for evaluating PCR positivity correlations with seroconversion and disease severity?
  Ожидаемый ответ: 'Retrospective cohort study'
  Semantic

In [21]:
import numpy as np
import math

docs = [chunk for chunks in data['chunks_no_overlap'] for chunk in chunks]
tokenized_docs = [doc.lower().split() for doc in docs]
doc_lengths = np.array([len(doc) for doc in tokenized_docs])
avgdl = np.mean(doc_lengths)
df = {}
for doc in tokenized_docs:
    for word in set(doc):
        df[word] = df.get(word, 0) + 1
        
N = len(docs)
idf = {word: math.log((N - freq + 0.5) / (freq + 0.5) + 1) for word, freq in df.items()}

def bm25_search(query, k=10):
    tokenized_query = query.lower().split()
    scores = np.zeros(N)
    
    for i, doc in enumerate(tokenized_docs):
        score = 0
        for word in tokenized_query:
            if word in idf:
                freq = doc.count(word)
                if freq > 0:
                    k1 = 1.5
                    b = 0.75
                    tf_norm = (freq * (k1 + 1)) / (freq + k1 * (1 - b + b * (doc_lengths[i] / avgdl)))
                    score += idf[word] * tf_norm
        scores[i] = score
        
    ranked_indices = np.argsort(scores)[::-1][:k]
    return list(ranked_indices)

doc_embeddings = st_model.encode(docs, convert_to_numpy=True)

def dense_search(query, k=10):
    query_emb = st_model.encode(query, convert_to_numpy=True)
    scores = np.dot(doc_embeddings, query_emb)
    ranked_indices = np.argsort(scores)[::-1][:k]
    return list(ranked_indices)

def hybrid_search(query, k=3):
    dense_res = dense_search(query, 10)
    bm25_res = bm25_search(query, 10)
    

    rrf_scores = {}
    for rank, idx in enumerate(dense_res):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1 / (60 + rank + 1)
    for rank, idx in enumerate(bm25_res):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1 / (60 + rank + 1)
        
    sorted_indices = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)
    return sorted_indices[:k]
#в сложных вопросах используются точные термины
hard_questions = [
    ("Which organization designated the coronavirus as a global pandemic?", "World Health Organization"),
    ("What clinical efficacy did vaccine candidates recently show to mitigate risk factors?", "95% clinical efficacy"),
    ("What was the study design for evaluating PCR positivity correlations with seroconversion and disease severity?", "Retrospective cohort study"),
    ("Five GABA(A)R mAbs from what fluid cells bound to various epitopes involving the α1?", "cerebrospinal fluid"),
    ("The epidemiological characteristics of respiratory viruses in 2020 were compared with which year?", "2019")
]

normal_questions = [
    ("What specimens were collected to detect common childhood respiratory viruses across different age groups?", "Respiratory specimens"),
    ("Why is differential diagnoses for enteritis in adult ruminants not simple based on clinical signs alone?", "zoonotic potential"),
    ("What phenotype with seizures and increased mortality was induced in rodents reminiscent of encephalitis patients' symptoms?", "severe phenotype"),
    ("What are healthcare systems using repurposing drugs for to cure patients from COVID-19?", "cure the patients"),
    ("What was the study design for the research on coronavirus disease 2019?", "Retrospective case series")
]

def evaluate(questions):
    recall_dense, recall_bm25, recall_hybrid = 0, 0, 0
    
    for q, expected in questions:
        d_res = dense_search(q, 3)
        b_res = bm25_search(q, 3)
        h_res = hybrid_search(q, 3)
        
        if any(expected.lower() in docs[i].lower() for i in d_res): recall_dense += 1
        if any(expected.lower() in docs[i].lower() for i in b_res): recall_bm25 += 1
        if any(expected.lower() in docs[i].lower() for i in h_res): recall_hybrid += 1
        
    return recall_dense/len(questions), recall_bm25/len(questions), recall_hybrid/len(questions)

d_h, b_h, h_h = evaluate(hard_questions)
d_n, b_n, h_n = evaluate(normal_questions)

print(f"Трудные вопросы:  Dense={d_h*100:.0f}%, BM25={b_h*100:.0f}%, Hybrid={h_h*100:.0f}%")
print(f"Обычные вопросы:  Dense={d_n*100:.0f}%, BM25={b_n*100:.0f}%, Hybrid={h_n*100:.0f}%")

Трудные вопросы:  Dense=40%, BM25=60%, Hybrid=60%
Обычные вопросы:  Dense=60%, BM25=80%, Hybrid=60%


In [22]:
def get_top_10(query):
    results = collection_no.query(query_texts=query, n_results=10)
    return results['documents'][0]

ks = [1, 3, 5, 10]
total_relevant = {k: 0 for k in ks}

for q, expected_answer in eval_data:
    top_10 = get_top_10(q)
    
    for k in ks:
        top_k = top_10[:k]
        relevant_count = sum(1 for chunk in top_k if expected_answer.lower() in chunk.lower())
        total_relevant[k] += relevant_count

for k in ks:
    precision = (total_relevant[k] / (k * len(eval_data))) * 100
    noise = 100 - precision
    print(f"При k={k:>2}: Полезный контекст = {precision:.0f}%, Шум (мусор) = {noise:.0f}%")

При k= 1: Полезный контекст = 40%, Шум (мусор) = 60%
При k= 3: Полезный контекст = 17%, Шум (мусор) = 83%
При k= 5: Полезный контекст = 14%, Шум (мусор) = 86%
При k=10: Полезный контекст = 7%, Шум (мусор) = 93%
